In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_operator_assignment_intervals AS
WITH 
-- 1. Rozbicie planów na dni kalendarzowe
expanded_plan_days AS (
  SELECT 
    p.plan_key,
    upper(trim(p.line_code)) AS line_code,
    p.start_datetime,
    p.end_datetime,
    CAST(p.start_datetime AS DATE) AS plan_start_date,
    CAST(p.end_datetime AS DATE) AS plan_end_date,
    date_format(p.start_datetime, 'HH:mm:ss') AS plan_start_time_str,
    date_format(p.end_datetime, 'HH:mm:ss') AS plan_end_time_str,
    explode(sequence(CAST(p.start_datetime AS DATE), CAST(p.end_datetime AS DATE), interval 1 day)) AS production_date
  FROM data_warehouse_factory.silver.silver_production_plan p
  WHERE p.line_code IS NOT NULL AND trim(p.line_code) != ''
),

-- 2. Wyznaczenie ram czasowych zmiany z planu
daily_plan_windows AS (
  SELECT 
    plan_key,
    line_code,
    production_date,
    CAST(date_format(production_date, 'yyyyMMdd') AS INT) AS date_key,
    
    to_timestamp(concat(cast(production_date as string), ' ', 
      CASE WHEN production_date = plan_start_date THEN plan_start_time_str ELSE '06:00:00' END
    )) AS daily_plan_start,
    
    to_timestamp(concat(cast(production_date as string), ' ', 
      CASE WHEN production_date = plan_end_date THEN plan_end_time_str ELSE '14:00:00' END
    )) AS daily_plan_end
  FROM expanded_plan_days
),

-- 3. Pobranie obsady domyślnej dla planowanych linii i komórek
planned_cells AS (
  SELECT
    dp.plan_key,
    dp.production_date AS start_date,
    dp.date_key,
    dp.line_code,
    a.cell_code,
    a.cell_name,
    dp.daily_plan_start AS plan_start,
    dp.daily_plan_end AS plan_end,
    a.default_employee_key,
    a.backup_employee_key
  FROM daily_plan_windows dp
  INNER JOIN data_warehouse_factory.silver.silver_employee_assignments a 
    ON dp.line_code = a.line_code
    AND dp.production_date >= a.valid_from 
    AND dp.production_date <= coalesce(a.valid_to, DATE '9999-12-31')
  WHERE dp.daily_plan_start < dp.daily_plan_end
  QUALIFY ROW_NUMBER() OVER(
    PARTITION BY dp.production_date, dp.line_code, a.cell_code 
    ORDER BY a.valid_from DESC, a._bronze_ingested_at DESC
  ) = 1
),

-- 4. Zastępstwa wchodzące (Swap-In)
fct_swap_roles_in AS (
  SELECT
    e.production_date AS start_date,
    upper(trim(e.cell_code)) AS cell_code,
    e.daily_event_start AS start_ts,
    e.daily_event_end AS end_ts,
    e.employee_key
  FROM data_warehouse_factory.silver.silver_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON upper(trim(e.event_type)) = upper(trim(d.event_type_name))
  WHERE d.is_swap = TRUE 
    AND e.cell_code IS NOT NULL 
    AND upper(trim(e.cell_code)) NOT IN ('NO ASSIGNMENT', 'NO_CELL')
),

-- 5. Oddelegowania pracownika macierzystego (Swap-Out)
fct_swap_roles_out AS (
  SELECT
    e.production_date AS start_date,
    e.employee_key AS swapped_out_operator_key,
    e.daily_event_start AS start_ts,
    e.daily_event_end AS end_ts
  FROM data_warehouse_factory.silver.silver_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON upper(trim(e.event_type)) = upper(trim(d.event_type_name))
  WHERE d.is_swap = TRUE
),

-- 6. Nieobecności pracownika (Absences)
absences AS (
  SELECT 
    e.production_date AS start_date,
    e.employee_key AS absent_operator_key,
    e.daily_event_start AS start_ts,
    e.daily_event_end AS end_ts
  FROM data_warehouse_factory.silver.silver_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON upper(trim(e.event_type)) = upper(trim(d.event_type_name))
  WHERE d.is_absence = TRUE
),

-- 7. SIATKA GNIAZD (GRID): Wszystkie gniazda z planu ORAZ wszystkie stacje ze zdarzeń Swap-In
grid_cells AS (
  SELECT start_date, cell_code FROM planned_cells
  UNION DISTINCT
  SELECT start_date, cell_code FROM fct_swap_roles_in
),

-- 8. Wzbogacenie siatki o domyślnych pracowników ze słownika (jeśli gniazdo miało przypisanie)
grid_with_context AS (
  SELECT 
    g.start_date,
    CAST(date_format(g.start_date, 'yyyyMMdd') AS INT) AS date_key,
    g.cell_code,
    COALESCE(p.line_code, a.line_code) AS line_code,
    p.plan_start,
    p.plan_end,
    COALESCE(p.default_employee_key, a.default_employee_key) AS default_employee_key,
    COALESCE(p.backup_employee_key, a.backup_employee_key) AS backup_employee_key
  FROM grid_cells g
  LEFT JOIN planned_cells p 
    ON g.start_date = p.start_date AND g.cell_code = p.cell_code
  LEFT JOIN data_warehouse_factory.silver.silver_employee_assignments a
    ON g.cell_code = a.cell_code
    AND g.start_date >= a.valid_from 
    AND g.start_date <= coalesce(a.valid_to, DATE '9999-12-31')
  QUALIFY ROW_NUMBER() OVER(
    PARTITION BY g.start_date, g.cell_code 
    ORDER BY a.valid_from DESC
  ) = 1
),

-- 9. Punkty graniczne osi czasu (Breakpoints)
all_time_points AS (
  -- Punkty z planu (tam, gdzie plan istniał)
  SELECT start_date, cell_code, plan_start AS bp_time FROM grid_with_context WHERE plan_start IS NOT NULL
  UNION DISTINCT
  SELECT start_date, cell_code, plan_end AS bp_time FROM grid_with_context WHERE plan_end IS NOT NULL
  
  -- Punkty ze Swap-In (również poza planem!)
  UNION DISTINCT
  SELECT s.start_date, s.cell_code, s.start_ts AS bp_time FROM fct_swap_roles_in s
  UNION DISTINCT
  SELECT s.start_date, s.cell_code, s.end_ts AS bp_time FROM fct_swap_roles_in s
  
  -- Punkty z oddelegowań i nieobecności operatora domyślnego
  UNION DISTINCT
  SELECT g.start_date, g.cell_code, so.start_ts AS bp_time
  FROM grid_with_context g
  INNER JOIN fct_swap_roles_out so 
    ON g.start_date = so.start_date AND g.default_employee_key = so.swapped_out_operator_key
  UNION DISTINCT
  SELECT g.start_date, g.cell_code, so.end_ts AS bp_time
  FROM grid_with_context g
  INNER JOIN fct_swap_roles_out so 
    ON g.start_date = so.start_date AND g.default_employee_key = so.swapped_out_operator_key

  UNION DISTINCT
  SELECT g.start_date, g.cell_code, a.start_ts AS bp_time
  FROM grid_with_context g
  INNER JOIN absences a 
    ON g.start_date = a.start_date AND g.default_employee_key = a.absent_operator_key
  UNION DISTINCT
  SELECT g.start_date, g.cell_code, a.end_ts AS bp_time
  FROM grid_with_context g
  INNER JOIN absences a 
    ON g.start_date = a.start_date AND g.default_employee_key = a.absent_operator_key
),

-- 10. Budowanie i weryfikacja interwałów
creating_intervals AS (
  SELECT 
    start_date,
    cell_code,
    bp_time AS interval_start,
    LEAD(bp_time) OVER (PARTITION BY start_date, cell_code ORDER BY bp_time) AS interval_end
  FROM all_time_points
  WHERE bp_time IS NOT NULL
),

-- 11. Zachowujemy interwał, jeśli mieści się w planie LUB nakłada się na Swap-In
valid_intervals AS (
  SELECT 
    i.start_date,
    g.date_key,
    g.line_code,
    i.cell_code,
    i.interval_start,
    i.interval_end,
    g.plan_start,
    g.plan_end,
    g.default_employee_key,
    g.backup_employee_key
  FROM creating_intervals i
  INNER JOIN grid_with_context g 
    ON i.start_date = g.start_date 
    AND i.cell_code = g.cell_code
  WHERE i.interval_end IS NOT NULL 
    AND i.interval_start < i.interval_end
    AND (
      -- Warunek 1: Miele się w nominalnym planie gniazda
      (g.plan_start IS NOT NULL AND i.interval_start >= g.plan_start AND i.interval_end <= g.plan_end)
      OR
      -- Warunek 2: Praca poza planem, ale pokryta przez Swap-In
      EXISTS (
        SELECT 1 FROM fct_swap_roles_in si
        WHERE si.start_date = i.start_date 
          AND si.cell_code = i.cell_code
          AND i.interval_start >= si.start_ts 
          AND i.interval_start < si.end_ts
      )
    )
)

-- 12. Wyliczenie ostatecznych przypisań i kategoryzacja
SELECT 
  md5(concat_ws('||', cast(v.date_key as string), v.cell_code, cast(v.interval_start as string))) AS interval_stage_key,
  v.date_key,
  v.start_date,
  v.line_code,
  v.cell_code,
  date_format(v.interval_start, 'HH:mm:ss') AS start_time,
  date_format(v.interval_end, 'HH:mm:ss') AS end_time,
  v.interval_start,
  v.interval_end,
  ROUND(timestampdiff(MINUTE, v.interval_start, v.interval_end) / 60.0, 2) AS duration_hours,
  ROUND(timestampdiff(MINUTE, v.interval_start, v.interval_end), 2) AS duration_minutes,
  
  -- Flaga informująca, czy interwał był w planie produkcji
  CASE 
    WHEN v.plan_start IS NOT NULL AND v.interval_start >= v.plan_start AND v.interval_end <= v.plan_end THEN TRUE 
    ELSE FALSE 
  END AS is_planned_window,

  -- Hierarchia przypisania
  COALESCE(
    si.employee_key, 
    CASE 
      WHEN a.absent_operator_key IS NOT NULL OR so.swapped_out_operator_key IS NOT NULL 
      THEN v.backup_employee_key 
      ELSE v.default_employee_key 
    END
  ) AS assigned_employee_key,

  -- Precyzyjny status aktywności
  CASE
    WHEN si.employee_key IS NOT NULL AND (v.plan_start IS NULL OR v.interval_start < v.plan_start OR v.interval_end > v.plan_end) 
      THEN 'SWAP_OUTSIDE_PLAN'
    WHEN si.employee_key IS NOT NULL 
      THEN 'SWAP_IN'
    WHEN a.absent_operator_key IS NOT NULL 
      THEN 'BACKUP_COVERING_ABSENCE'
    WHEN so.swapped_out_operator_key IS NOT NULL 
      THEN 'BACKUP_COVERING_SWAP_OUT'
    WHEN v.default_employee_key IS NOT NULL 
      THEN 'DEFAULT_ASSIGNMENT'
    ELSE 'UNASSIGNED'
  END AS assignment_status,
  
  v.default_employee_key,
  v.backup_employee_key,
  si.employee_key AS swap_in_employee_key,
  so.swapped_out_operator_key,
  a.absent_operator_key,
  
  CURRENT_TIMESTAMP() AS _silver_created_at

FROM valid_intervals v
LEFT JOIN fct_swap_roles_in si 
  ON v.start_date = si.start_date 
  AND v.cell_code = si.cell_code 
  AND v.interval_start >= si.start_ts 
  AND v.interval_start < si.end_ts

LEFT JOIN absences a 
  ON v.start_date = a.start_date 
  AND v.default_employee_key = a.absent_operator_key
  AND v.interval_start >= a.start_ts 
  AND v.interval_start < a.end_ts

LEFT JOIN fct_swap_roles_out so
  ON v.start_date = so.start_date
  AND v.default_employee_key = so.swapped_out_operator_key
  AND v.interval_start >= so.start_ts
  AND v.interval_start < so.end_ts;